# 04 — The editing experiment (Figures 7 & 8)
Three ablation curves on the adversarial test set:
- `text-sorted` — ablate MILAN-flagged text neurons, smallest val-acc impact first.
- `sort-all` — baseline: ablate any neuron with smallest val-acc impact.
- `random` — averaged over 5 random orderings.

If MILAN is doing useful interpretability work, `text-sorted` should rise above the random and (most of) the sort-all curves on adversarial accuracy.

In [ ]:
import os, sys, torch
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

version_dir = Path(os.environ['MILAN_DATA_DIR']) / 'imagenet-spurious-text' / '50pct'
ckpt = Path(os.environ['MILAN_MODELS_DIR']) / 'resnet18_spurious.pth'
results = Path(os.environ['MILAN_RESULTS_DIR'])
dissect_dir = results / 'edit' / 'imagenet-spurious-text' / 'resnet18_spurious-50pct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from milan_repro.editing.evaluate import run as run_editing
run_editing(version_dir, ckpt, dissect_dir,
            results / 'descriptions_annotated.csv',
            results / 'ablation_curve.csv',
            n_random_trials=5, ablation_max=50, ablation_step=2,
            device=device)

In [ ]:
from milan_repro.figures.plot_fig7 import plot as plot_fig7
from milan_repro.figures.plot_fig8 import plot as plot_fig8
plot_fig7(dissect_dir, results / 'descriptions_annotated.csv',
          results / 'figs' / 'fig7.pdf')
plot_fig8(results / 'ablation_curve.csv', results / 'figs' / 'fig8.pdf')

In [ ]:
import pandas as pd
df = pd.read_csv(results / 'ablation_curve.csv')
best = (df[df['mode'].isin(['text-sorted', 'sort-all'])]
        .groupby('mode')['adv_acc'].max())
baseline = df[df['mode'] == 'baseline'].iloc[0]
print('baseline adv acc:', baseline['adv_acc'])
print('best per mode:')
print(best)